In [0]:
# Acesso ao config


%run "/Workspace/Repos/luizhpdatasci@gmail.com/merca-data-platform/Squad3/luiz-portacio/config/00_config.ipynb"

In [0]:
# ══════════════════════════════════════
# 00_UTILS — Funções Utilitárias
# Squad 3 — Batch Ecommerce
# ══════════════════════════════════════

# Listar Arquivos do Container

import os
import pandas as pd
from io import BytesIO
from dotenv import load_dotenv
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient

load_dotenv()

print("✅ Imports carregados!")

In [0]:
# Listar Arquivos do Container

def listar_arquivos(adls_client, container):
    fs_client = adls_client.get_file_system_client(file_system=container)
    arquivos  = [p.name for p in fs_client.get_paths() if not p.is_directory]

    print(f"📁 Arquivos encontrados: {len(arquivos)}\n")
    for arquivo in arquivos:
        print(f"   - {arquivo}")

    return arquivos

print("✅ Função listar_arquivos criada!")


In [0]:
# Funções de Leitura

def ler_csv(adls_client, container, caminho_arquivo, separador=","):
    """
    Lê arquivos CSV pequenos via SDK Azure.
    Recomendado para arquivos até 50MB.
    """
    fs_client   = adls_client.get_file_system_client(file_system=container)
    file_client = fs_client.get_file_client(caminho_arquivo)
    conteudo    = file_client.download_file().readall()
    return pd.read_csv(BytesIO(conteudo), sep=separador)


def ler_parquet(adls_client, container, caminho_arquivo):
    """
    Lê arquivos Parquet pequenos via SDK Azure.
    Recomendado para arquivos até 50MB.
    """
    fs_client   = adls_client.get_file_system_client(file_system=container)
    file_client = fs_client.get_file_client(caminho_arquivo)
    conteudo    = file_client.download_file().readall()
    return pd.read_parquet(BytesIO(conteudo))


def ler_csv_chunks(adls_client, container, caminho_arquivo, chunk_size=50000):
    """
    Lê arquivos CSV grandes em partes (chunks) via SDK Azure.
    Recomendado para arquivos acima de 50MB.
    chunk_size = quantidade de linhas por chunk (padrão: 50.000)
    """
    fs_client   = adls_client.get_file_system_client(file_system=container)
    file_client = fs_client.get_file_client(caminho_arquivo)
    conteudo    = file_client.download_file().readall()

    chunks   = pd.read_csv(BytesIO(conteudo), chunksize=chunk_size)
    df_final = pd.concat(chunks, ignore_index=True)

    print(f"✅ Arquivo lido em chunks!")
    print(f"   Linhas : {df_final.shape[0]}")
    print(f"   Colunas: {df_final.shape[1]}")
    return df_final

print("✅ Funções ler_csv, ler_parquet e ler_csv_chunks criadas!")

In [0]:
# Funções de Escrita no SQL Server

def salvar_tabela(df, nome_tabela, modo="overwrite"):
    """
    Salva DataFrame no SQL Server via Spark.
    Recomendado para arquivos até 50MB.
    """
    options  = get_sql_options()
    df_spark = spark.createDataFrame(df)

    df_spark.write \
        .format("sqlserver") \
        .option("host", options["host"]) \
        .option("port", options["port"]) \
        .option("database", options["database"]) \
        .option("user", options["user"]) \
        .option("password", options["password"]) \
        .option("dbtable", f"squad3.{nome_tabela}") \
        .option("encrypt", options["encrypt"]) \
        .option("trustServerCertificate", options["trustServerCertificate"]) \
        .mode(modo) \
        .save()

    print(f"✅ Tabela squad3.{nome_tabela} salva! ({len(df)} linhas)")


def salvar_tabela_lotes(df, nome_tabela, lote_size=5000, modo="overwrite", max_tentativas=3):
    import time

    options = get_sql_options()
    total   = len(df)
    lotes   = range(0, total, lote_size)

    print(f"📦 Salvando {total} linhas em lotes de {lote_size}...")
    print(f"   Total de lotes: {len(lotes)}\n")

    for i, inicio in enumerate(lotes):
        fim       = min(inicio + lote_size, total)
        lote      = df.iloc[inicio:fim]
        modo_lote = modo if i == 0 else "append"

        for tentativa in range(1, max_tentativas + 1):
            try:
                df_spark = spark.createDataFrame(lote)

                df_spark.write \
                    .format("sqlserver") \
                    .option("host", options["host"]) \
                    .option("port", options["port"]) \
                    .option("database", options["database"]) \
                    .option("user", options["user"]) \
                    .option("password", options["password"]) \
                    .option("dbtable", f"squad3.{nome_tabela}") \
                    .option("encrypt", options["encrypt"]) \
                    .option("trustServerCertificate", options["trustServerCertificate"]) \
                    .mode(modo_lote) \
                    .save()

                print(f"   ✅ Lote {i+1}/{len(lotes)} — linhas {inicio} a {fim}")
                break

            except Exception as e:
                print(f"   ⚠️  Lote {i+1} — tentativa {tentativa}/{max_tentativas} falhou: {e}")

                if tentativa < max_tentativas:
                    print(f"   🔄 Aguardando 10s antes de tentar novamente...")
                    time.sleep(10)
                else:
                    print(f"   ❌ Lote {i+1} falhou após {max_tentativas} tentativas.")
                    raise

    print(f"\n✅ Tabela squad3.{nome_tabela} salva! ({total} linhas)")

def consultar_tabela(nome_tabela):
    """
    Lê uma tabela do SQL Server e retorna como DataFrame pandas.
    """
    options = get_sql_options()

    df_resultado = spark.read \
        .format("sqlserver") \
        .option("host", options["host"]) \
        .option("port", options["port"]) \
        .option("database", options["database"]) \
        .option("user", options["user"]) \
        .option("password", options["password"]) \
        .option("dbtable", f"squad3.{nome_tabela}") \
        .option("encrypt", options["encrypt"]) \
        .option("trustServerCertificate", options["trustServerCertificate"]) \
        .load()

    return df_resultado.toPandas()

print("✅ Funções salvar_tabela, salvar_tabela_lotes e consultar_tabela criadas!")

In [0]:
# Teste JDBC (CREATE/INSERT)

def testar_conexao_sql():
    options = get_sql_options()

    df_teste = spark.createDataFrame(
        [(1, "Conexão com sucesso")],
        ["id", "mensagem"]
    )

    df_teste.write \
        .format("sqlserver") \
        .option("host", options["host"]) \
        .option("port", options["port"]) \
        .option("database", options["database"]) \
        .option("user", options["user"]) \
        .option("password", options["password"]) \
        .option("dbtable", "squad3.teste_conexao") \
        .option("encrypt", options["encrypt"]) \
        .option("trustServerCertificate", options["trustServerCertificate"]) \
        .mode("overwrite") \
        .save()

    print("✅ Conexão SQL Server testada!")
    print(f"   Permissão de CREATE : ✅")
    print(f"   Permissão de INSERT : ✅")

testar_conexao_sql()
print("✅ 00_utils carregado com sucesso!")